In [1]:
from typesafe_sdk import TypeSafeClient, Noul, Score
from dotenv import load_dotenv
from utility.tools.pg_tools import direct_db_access
from utility.database.sql_utilities import sql_literal
import pandas as pd

load_dotenv()

client = TypeSafeClient()

In [2]:
course_id = "4f5ccc0da82e4a77b61e70886e6e247b"
retuned_data = direct_db_access(
    f"select review, review_id from reviews.course_reviews where course_id={sql_literal(course_id)}"
)

In [3]:
reviews_flatted = [i[0] for i in retuned_data[1]]

In [4]:
questions = {
    "curriculum_mentioned": Noul(
        instructions="Does the review mentions about the course curriculum?"
    ),
    "about_instructor_mentioned": Noul(
        instructions="Does the review mentions about the course instructor/trainer and it's credibility?"
    ),
    "hands_on_work_mentioned": Noul(
        instructions="Does the review mentions about the hands on and practial work in course?"
    ),
    "mentorship_support_mentioned": Noul(
        instructions="Does the review mentions about the mentorship and support provided in the course?"
    ),
    "career_outcome_mentioned": Noul(
        instructions="Does the review mentions about the carrer outcomes from course?"
    ),
    "learning_experience_mentioned": Noul(
        instructions="Does the review mentions about the learning experience in the course?"
    ),
    "cost_price_worth_mentioned": Noul(
        instructions="Does the review mentions about the cost, investment or quantative outcomes from the course?"
    ),
    "flexibility_mentioned": Noul(
        instructions="Does the review mentions about the flexiility and schedules in the course?"
    ),
    "industry_relevance_mentioned": Noul(
        instructions="Does the review mentions about industry relevance of the course?"
    ),
}

In [5]:
satisfaction_levels = [
    "Very dissatisfied",
    "Dissatisfied",
    "Neither satisfied nor dissatisfied, or balanced mixed feedback",
    "Satisfied",
    "Very satisfied",
    "Exceptional / beyond expectations",
]

In [6]:
reviews_question_answered = []
for review in reviews_flatted:
    state = {"query": review}
    response = client.system_one(state=state, questions=questions)
    reviews_question_answered.append((review, response.answers))

In [7]:
review_selected_per_review = {}
for review_with_answer in reviews_question_answered:
    for question, answer in review_with_answer[1].items():
        if answer.noul > 0.7:
            if question not in review_selected_per_review.keys():
                review_selected_per_review[question] = [review_with_answer[0]]
            else:
                review_selected_per_review[question].append(review_with_answer[0])

In [8]:
selected_review_scoring = {}

for question, selected_reviews in review_selected_per_review.items():
    selected_review_scoring[question] = {"reviews": [], "scores": []}
    question_for_scoring = {
        f"{question}_score": Score(
            instructions=f"with respect to {question}, the review lies under which category?",
            criteria=satisfaction_levels,
        )
    }
    for review in selected_reviews:
        response = client.system_one(state=review, questions=question_for_scoring)
        selected_review_scoring[question]["reviews"].append(review)
        selected_review_scoring[question]["scores"].append(response.answers)

In [9]:
df = pd.DataFrame(
    {
        "review_id": [i[1] for i in retuned_data[1]],
        "review": [i[0] for i in retuned_data[1]],
    }
)

for aspect, block in selected_review_scoring.items():
    score_map = {
        review: score_dict[f"{aspect}_score"].score
        for review, score_dict in zip(block["reviews"], block["scores"])
    }
    df[aspect] = df["review"].map(score_map)

df = df.rename(columns=lambda c: c.replace("_mentioned", ""))
df.to_csv("check.csv", index=False)

In [10]:
df

,review_id,review,about_instructor,learning_experience,cost_price_worth,industry_relevance,curriculum,hands_on_work,mentorship_support
0,413c2b0878c3019d431ca6d9163f2fa239283aa3d53e5b...,"I am on 11th day of this course, 2 more days o...",4.61,4.59,4.29,4.36,NaN,NaN,NaN
1,93373313164b3c75e93489ebe5a0a7fc0aed1f44bc530a...,Covered all important topics of snowflake and ...,NaN,4.17,NaN,NaN,4.16,NaN,NaN
2,c1b05adb1fa59a071c74f902c2ad1362a1a94bfdbe721e...,Ankit has explained the all the topics in deta...,NaN,4.26,NaN,NaN,NaN,NaN,NaN
3,d9d9fb644b3159f1d9ff514c6790f15ad86e267684ff6c...,"This bootcamp was well-structured, and all the...",NaN,4.27,NaN,NaN,4.17,4.10,NaN
4,3734f84aa37c2d34860f78c63ebd935e70b738b4ef6748...,The Course was amazing. Similar to other video...,4.19,4.17,NaN,NaN,4.17,NaN,3.85
5,bb15ebef784bd6213f68890a0b43e6a0976a9b25b1014c...,This is an excellent course for anyone startin...,NaN,4.47,NaN,NaN,4.37,4.31,NaN
6,682235a7ca61bb736ab5b779b76659b6d9e41786bbfe4e...,The course was a great hands-on learning exper...,NaN,4.18,NaN,NaN,4.14,4.25,NaN
7,09bed3efb3a12e970c11d502668a1b4f783a0d7a6cc95a...,This course was incredibly well-structured and...,NaN,4.26,NaN,NaN,4.09,NaN,NaN
8,76c2ac65a9c6bc91e24ddb691dd74e4546cdaa154e1d88...,The course offered a strong foundation and cle...,4.31,4.14,NaN,NaN,3.92,3.95,4.13
9,267ddf1cf51739678c66db264acbe3ab551d28bd86e286...,"When I started this course, I knew nothing abo...",4.04,4.13,NaN,3.90,NaN,NaN,NaN
